In [2]:
import sys
from pathlib import Path

# resolve imports from the current working dir.
cwd = Path.cwd().resolve()
project_root = cwd.parent
sys.path.insert(0, str(project_root))

from FEX.utils.numerical_deriv import NumericalDeriv
from FEX import (
    ControllerConfig,
    FEXConfig,
    train_network_controller,
    runtimeconfig,
    get_tree_config
)

import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

Using device: cuda


In [3]:
import pandas as pd
import numpy as np

adj_matrix = pd.read_csv('data/BA_Nnodes100_Adj.csv', header=None)
num_graph_nodes = adj_matrix.shape[0]

x_df = pd.read_csv('data/HR_timeseries_SNR_40.csv', header=None)
num_timesteps, num_cols = x_df.shape
x_np = x_df.to_numpy(dtype=np.float32)
x_data = torch.from_numpy(x_np.reshape(num_timesteps, num_graph_nodes, 3))

dt = 0.01
dx_dt = NumericalDeriv(x_data, dt=dt) # 4th order
x_data = x_data[2:-2]

In [ ]:
# interactive 3d trajectory for the first node in x_data
import plotly.graph_objects as go

coords = x_data[:, 10, :].detach().cpu().numpy()
bounds = np.where((coords[:, 2] < 4))
coords = coords[bounds]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=coords[:, 0],
            y=coords[:, 1],
            z=coords[:, 2],
            mode='lines',
            line=dict(width=2),
        )
    ]
    
)

fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
    ),
    margin=dict(l=0, r=0, b=0, t=10),
)

fig.show()


In [ ]:
train_test_split = int(0.8 * len(x_data))
train_x_data = x_data[:train_test_split]
train_dx_dt = dx_dt[:train_test_split]
adj_matrix_tensor = torch.tensor(adj_matrix.values, dtype=torch.float32).to(runtimeconfig.device)
x_data_tensor_ds = TensorDataset(train_x_data, train_dx_dt)
if runtimeconfig.device == "cuda":
    dataloader = DataLoader(x_data_tensor_ds, batch_size=32, shuffle=True, pin_memory=True)
else:
    dataloader = DataLoader(x_data_tensor_ds, batch_size=32, shuffle=True)

In [ ]:
from FEX.utils.tree_configs import *
tree_config = get_tree_config("depth_3_leaves_4_config")
controller_config = ControllerConfig(
    input_dim = 20,
    hidden_dim = 64,
    lr = 0.001,
    num_epochs = 100,
    num_cands_per_epoch = 10,
    percentile_threshold = 0.4,
    num_trees = 2
)
fex_config = FEXConfig(
    num_epochs = 20, 
    lr = 0.002,
    bfgs_lr = 0.1,
    max_nodes = 20,
    max_norm = 1.0,
    leaf_dim = x_data.shape[2],

    num_leaves = tree_config.num_leaves
)

best_candidates = train_network_controller(
    tree_config,
    dataloader,
    adj_matrix_tensor,
    controller_config,
    fex_config,
)

2026-03-21 04:35:34,018 | WARNING | Adam Training Completed with non-finite loss: nan
2026-03-21 04:35:34,021 | INFO | Epoch 0, Candidate 0, Score: 2867811109206729195322347618304.0000, Reward: 0.0000
2026-03-21 04:35:36,278 | WARNING | Adam Training Completed with non-finite loss: nan
2026-03-21 04:35:36,280 | INFO | Epoch 0, Candidate 1, Score: 1004967100416.0000, Reward: 0.0000
2026-03-21 04:35:38,507 | WARNING | Adam Training Completed with non-finite loss: nan
2026-03-21 04:35:38,509 | WARNING | Epoch 0, Candidate 2 produced non-finite score; assigning zero reward.
2026-03-21 04:35:38,510 | INFO | Epoch 0, Candidate 2, Score: inf, Reward: 0.0000
2026-03-21 04:35:40,612 | WARNING | Adam Training Completed with non-finite loss: nan
2026-03-21 04:35:40,613 | WARNING | Epoch 0, Candidate 3 produced non-finite score; assigning zero reward.
2026-03-21 04:35:40,614 | INFO | Epoch 0, Candidate 3, Score: inf, Reward: 0.0000
2026-03-21 04:35:42,468 | WARNING | Adam Training Completed with n

In [6]:
best_candidates.visualize_candidates(directory="logs/candidates_viz")

In [7]:
best_candidates.save_candidates("logs/best_candidates")